<a href="https://colab.research.google.com/github/hmnaz213/native/blob/main/tenet_masaud_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# TENET V2 - BLOCK 02
# LOAD QUOTE-STAGE JSON
# ============================================


from google.colab import drive
drive.mount("/content/drive", force_remount=True)


from pathlib import Path
import json
import pandas as pd


QUOTE_JSON = Path("/content/drive/MyDrive/2026/tenet/quotes/masaud_quote_v2.json")


if not QUOTE_JSON.exists():
    raise FileNotFoundError(f"Quote JSON not found: {QUOTE_JSON}")


with open(QUOTE_JSON, "r", encoding="utf-8") as f:
    quote_json = json.load(f)


business_context = quote_json["business_context"]
quote_identity = quote_json["quote_identity"]
client = quote_json["client"]
quote = quote_json["quote"]
scope = quote_json["scope"]


line_items_df = pd.DataFrame(quote_json["line_items"])
cluster_summary_df = pd.DataFrame(quote_json["cluster_summary"])
financial_summary_df = pd.DataFrame([quote_json["financial_summary"]])


print("QUOTE JSON LOADED")
print("Quote ID:", quote_identity["quote_id"])
print("Client:", client["client_name"])


display(line_items_df)
display(cluster_summary_df)
display(financial_summary_df)



Mounted at /content/drive
QUOTE JSON LOADED
Quote ID: Q-20260411-0000
Client: Masaud


,line_id,description,unit_amount_ghs,quantity,amount_ghs,remarks,category,cluster
0,L1,700W Monofacial Solar Panel,1348.470200,15.000,20227.053000,,pv_module,cluster_1_pv_mounting
1,L2,PV mounting system,261.899771,49.200,12885.468750,,mounting_system,cluster_1_pv_mounting
2,L3,2 in 2 out DC Combiner (dual MPPT),447.747300,1.000,447.747300,,bos_cable,cluster_2_bos_cable
3,L4,2 in 2 out AC Box (Three Phase),482.189400,1.000,482.189400,,bos_cable,cluster_2_bos_cable
4,L5,1x6mm² PV Cable (m),847.275660,1.815,1537.805323,,bos_cable,cluster_2_bos_cable
5,L6,CIF Shipping of general goods to Tema Port,3103.330230,1.000,3103.330230,,shipping_general_goods,cluster_3_cif_general_goods_tema
6,L7,CIF Shipping of battery goods to Tema Port,5659.500000,1.000,5659.500000,,shipping_battery_goods,cluster_4_cif_battery_goods_tema
7,L8,Local transport to site,370.440000,1.000,370.440000,,local_transport,local_transport
8,L9,Installation & commissioning,3751.440000,1.000,3751.440000,,installation_commissioning,installation_commissioning


,cluster,cluster_amount_ghs
0,cluster_1_pv_mounting,33112.521750
1,cluster_2_bos_cable,2467.742023
2,cluster_3_cif_general_goods_tema,3103.330230
3,cluster_4_cif_battery_goods_tema,5659.500000
4,installation_commissioning,3751.440000
5,local_transport,370.440000


,quoted_total_ghs,quoted_total_from_lines_ghs,quoted_total_from_sheet_ghs
0,48464.974003,48464.974003,48464.974003


In [2]:
# ============================================
# MASAUD V2 - BLOCK 03
# LOAD CLIENT PAYMENTS FROM TENET CASH FLOW GSHEET
# ============================================


!pip -q install gspread gspread-dataframe


from google.colab import auth
auth.authenticate_user()


import gspread
from google.auth import default
from gspread_dataframe import get_as_dataframe
import pandas as pd


# ------------------------------------------------
# 1. INPUTS
# ------------------------------------------------
TENET_CASH_FLOW_URL = "https://docs.google.com/spreadsheets/d/1qT87DDH2sU35ytPN24aSEN0T4M86qji-YyL0fuj7nw8/edit?gid=2002674292#gid=2002674292"
PAYMENT_TAB = "Client Payments"
PROJECT_ID = "masaud_residence"


# ------------------------------------------------
# 2. AUTHORIZE + LOAD
# ------------------------------------------------
creds, _ = default()
gc = gspread.authorize(creds)


sh = gc.open_by_url(TENET_CASH_FLOW_URL)
ws = sh.worksheet(PAYMENT_TAB)


payments_df = get_as_dataframe(
    ws,
    evaluate_formulas=True,
    header=0,
    dtype=str
)


payments_df = payments_df.dropna(how="all").copy()
payments_df.columns = [str(c).strip().lower() for c in payments_df.columns]


# ------------------------------------------------
# 3. CLEAN NUMERIC FIELDS
# ------------------------------------------------
for col in ["amount_ghs", "balance_after_tx"]:
    if col in payments_df.columns:
        payments_df[col] = (
            payments_df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
        )
        payments_df[col] = pd.to_numeric(payments_df[col], errors="coerce")


# ------------------------------------------------
# 4. FILTER PROJECT PAYMENTS
# ------------------------------------------------
project_payments_df = payments_df[
    payments_df["project_id"].astype(str).str.lower().str.strip() == PROJECT_ID
].copy()


total_client_payments_ghs = project_payments_df["amount_ghs"].sum()


# ------------------------------------------------
# 5. DISPLAY
# ------------------------------------------------
print("ALL CLIENT PAYMENTS")
display(payments_df)


print(f"\n{PROJECT_ID.upper()} CLIENT PAYMENTS")
display(project_payments_df)


print(f"\nTotal client payments received: GHS {total_client_payments_ghs:,.2f}")


# ------------------------------------------------
# 6. KEEP IN MEMORY
# ------------------------------------------------
masaud_client_payments_df = project_payments_df.copy()
masaud_total_client_payments_ghs = total_client_payments_ghs


print("\nDataFrames ready:")
print("- masaud_client_payments_df")
print("- masaud_total_client_payments_ghs")



ALL CLIENT PAYMENTS


,email_date,project_id,direction,amount_ghs,description,transaction_datetime,balance_after_tx,email_subject,gmail_message_id,source
0,24/04/2026 14:31:47,masaud_residence,cash_in,15000,AL Pay MOHAMMED NAZIF HABIB LTU0000,2026-04-23 13:18,25869.43,Tenet Client Payments,19dbfe7634dfd8bd,gmail_sms
1,24/04/2026 14:31:23,masaud_residence,cash_in,9900,AL Pay MOHAMMED NAZIF HABIB LTU0000,2026-04-23 13:11,10869.43,Tenet Client Payments,19dbfe7067e09eef,gmail_sms
2,24/04/2026 14:30:51,masaud_residence,cash_in,100,AL Pay MOHAMMED NAZIF HABIB LTU0000,2026-04-23 10:20,969.43,Tenet Client Payment,19dbfe689d74ddb8,gmail_sms
3,29/04/2026 13:52:59,masaud_residence,cash_in,5000,AL Pay MOHAMMED NAZIF HABIB LTU0000,2026-04-29 12:10,5640.94,Tenet Client Payments,19dd983af96a34d6,gmail_sms



MASAUD_RESIDENCE CLIENT PAYMENTS


,email_date,project_id,direction,amount_ghs,description,transaction_datetime,balance_after_tx,email_subject,gmail_message_id,source
0,24/04/2026 14:31:47,masaud_residence,cash_in,15000,AL Pay MOHAMMED NAZIF HABIB LTU0000,2026-04-23 13:18,25869.43,Tenet Client Payments,19dbfe7634dfd8bd,gmail_sms
1,24/04/2026 14:31:23,masaud_residence,cash_in,9900,AL Pay MOHAMMED NAZIF HABIB LTU0000,2026-04-23 13:11,10869.43,Tenet Client Payments,19dbfe7067e09eef,gmail_sms
2,24/04/2026 14:30:51,masaud_residence,cash_in,100,AL Pay MOHAMMED NAZIF HABIB LTU0000,2026-04-23 10:20,969.43,Tenet Client Payment,19dbfe689d74ddb8,gmail_sms
3,29/04/2026 13:52:59,masaud_residence,cash_in,5000,AL Pay MOHAMMED NAZIF HABIB LTU0000,2026-04-29 12:10,5640.94,Tenet Client Payments,19dd983af96a34d6,gmail_sms



Total client payments received: GHS 30,000.00

DataFrames ready:
- masaud_client_payments_df
- masaud_total_client_payments_ghs


In [5]:
# ============================================
# MASAUD V2 - BLOCK 04
# QUOTE JSON → PROJECT KICKOFF JSON + GSHEET
# ============================================


!pip -q install gspread gspread-dataframe


from google.colab import auth
auth.authenticate_user()


import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe
import pandas as pd
import json
import re
from pathlib import Path
from datetime import datetime
import numpy as np


# ------------------------------------------------
# 1. INPUTS
# ------------------------------------------------
QUOTE_JSON_PATH = Path("/content/drive/MyDrive/2026/tenet/quotes/masaud_quote_v2.json")


GSHEET_URL = "https://docs.google.com/spreadsheets/d/1cItvZGeYZi0seQEMKSnqhlXgqNlmuMuTJAlKUSGrwAA/edit?gid=0#gid=0"


# ------------------------------------------------
# 2. HELPERS
# ------------------------------------------------
def slugify(x):
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    return x.strip("_")


def convert_numpy_to_python(obj):
    if isinstance(obj, dict):
        return {k: convert_numpy_to_python(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_to_python(v) for v in obj]
    elif isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


# ------------------------------------------------
# 3. LOAD QUOTE JSON
# ------------------------------------------------
if not QUOTE_JSON_PATH.exists():
    raise FileNotFoundError(f"Quote JSON not found: {QUOTE_JSON_PATH}")


with open(QUOTE_JSON_PATH, "r", encoding="utf-8") as f:
    quote_json = json.load(f)


client_name = quote_json["client"]["client_name"]
quote_id = quote_json["quote_identity"]["quote_id"]


# ------------------------------------------------
# 4. GENERATE PROJECT ID AT KICKOFF
# ------------------------------------------------
PROJECT_ID = f"{slugify(client_name)}_residence"
PROJECT_NAME = f"{client_name} Residence"


PROJECT_DIR = Path(f"/content/drive/MyDrive/2026/tenet/projects/{PROJECT_ID}")
SSOT_DIR = PROJECT_DIR / "ssot"
SSOT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------
# 5. CREATE PROJECT QUOTE JSON
# ------------------------------------------------
project_quote_json = quote_json.copy()


project_quote_json["project"] = {
    "project_id": PROJECT_ID,
    "project_name": PROJECT_NAME,
    "project_status": "active",
    "source_quote_id": quote_id,
    "kickoff_date": datetime.now().date().isoformat()
}


project_quote_json["lifecycle"]["is_project_created"] = True
project_quote_json["lifecycle"]["project_id"] = PROJECT_ID
project_quote_json["lifecycle"]["project_created_at"] = datetime.now().isoformat()


PROJECT_QUOTE_JSON_PATH = SSOT_DIR / "project_quote.json"


with open(PROJECT_QUOTE_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(convert_numpy_to_python(project_quote_json), f, indent=2, ensure_ascii=False)


# ------------------------------------------------
# 6. CORE PAYMENT METRICS
# ------------------------------------------------
quoted_total_ghs = float(project_quote_json["financial_summary"]["quoted_total_ghs"])
client_paid_ghs = float(masaud_total_client_payments_ghs)


remaining_to_collect_ghs = quoted_total_ghs - client_paid_ghs
collection_rate_percent = round((client_paid_ghs / quoted_total_ghs * 100), 2) if quoted_total_ghs else 0


# ------------------------------------------------
# 7. DASHBOARD TABLES
# ------------------------------------------------
cluster_df = pd.DataFrame(project_quote_json["cluster_summary"])
cluster_df["project_id"] = PROJECT_ID
cluster_df["quote_id"] = quote_id
cluster_df["quoted_total_ghs"] = quoted_total_ghs
cluster_df["client_paid_ghs"] = client_paid_ghs
cluster_df["remaining_to_collect_ghs"] = remaining_to_collect_ghs
cluster_df["collection_rate_percent"] = collection_rate_percent


line_items_df = pd.DataFrame(project_quote_json["line_items"])
line_items_df["project_id"] = PROJECT_ID
line_items_df["quote_id"] = quote_id
line_items_df["client_paid_ghs"] = client_paid_ghs
line_items_df["collection_rate_percent"] = collection_rate_percent


summary_df = pd.DataFrame([{
    "business_unit_id": project_quote_json["business_context"]["business_unit_id"],
    "portfolio_id": project_quote_json["business_context"]["portfolio_id"],
    "program_id": project_quote_json["business_context"]["program_id"],
    "quote_id": quote_id,
    "project_id": PROJECT_ID,
    "project_name": PROJECT_NAME,
    "client_name": client_name,
    "project_status": "active",
    "quoted_total_ghs": quoted_total_ghs,
    "client_paid_ghs": client_paid_ghs,
    "remaining_to_collect_ghs": remaining_to_collect_ghs,
    "collection_rate_percent": collection_rate_percent,
    "created_at": datetime.now().isoformat()
}])


# ------------------------------------------------
# 8. CREATE KICKOFF JSON
# ------------------------------------------------
kickoff_json = {
    "business_context": project_quote_json["business_context"],
    "quote_identity": project_quote_json["quote_identity"],
    "project": project_quote_json["project"],
    "financials": {
        "quoted_total_ghs": quoted_total_ghs,
        "client_paid_ghs": client_paid_ghs,
        "remaining_to_collect_ghs": remaining_to_collect_ghs,
        "collection_rate_percent": collection_rate_percent
    },
    "clusters": cluster_df.to_dict(orient="records"),
    "line_items": line_items_df.to_dict(orient="records"),
    "payments": masaud_client_payments_df.to_dict(orient="records"),
    "metadata": {
        "generated_at": datetime.now().isoformat(),
        "stage": "project_kickoff"
    }
}


KICKOFF_JSON_PATH = SSOT_DIR / "project_kickoff_summary.json"


with open(KICKOFF_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(convert_numpy_to_python(kickoff_json), f, indent=2, ensure_ascii=False)


# ------------------------------------------------
# 9. WRITE TO GSHEET
# ------------------------------------------------
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open_by_url(GSHEET_URL)


def write_sheet(tab_name, df):
    try:
        ws = sh.worksheet(tab_name)
        ws.clear()
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=tab_name, rows=500, cols=50)


    set_with_dataframe(ws, df, include_index=False)


write_sheet("project_summary", summary_df)
write_sheet("project_clusters", cluster_df)
write_sheet("project_line_items", line_items_df)
write_sheet("project_payments", masaud_client_payments_df)


# ------------------------------------------------
# 10. DISPLAY
# ------------------------------------------------
print("PROJECT CREATED FROM QUOTE")
print("Project ID:", PROJECT_ID)


print("\nProject quote JSON saved:")
print(PROJECT_QUOTE_JSON_PATH)


print("\nKickoff summary JSON saved:")
print(KICKOFF_JSON_PATH)


print("\nGSHEET UPDATED FOR DATA STUDIO")


print("\nSUMMARY")
display(summary_df)


print("\nCLUSTERS")
display(cluster_df)


print("\nLINE ITEMS")
display(line_items_df)


print("\nPAYMENTS")
display(masaud_client_payments_df)



PROJECT CREATED FROM QUOTE
Project ID: masaud_residence

Project quote JSON saved:
/content/drive/MyDrive/2026/tenet/projects/masaud_residence/ssot/project_quote.json

Kickoff summary JSON saved:
/content/drive/MyDrive/2026/tenet/projects/masaud_residence/ssot/project_kickoff_summary.json

GSHEET UPDATED FOR DATA STUDIO

SUMMARY


,business_unit_id,portfolio_id,program_id,quote_id,project_id,project_name,client_name,project_status,quoted_total_ghs,client_paid_ghs,remaining_to_collect_ghs,collection_rate_percent,created_at
0,tenet,renewable_energy,residential_solar_pv_bess_2026,Q-20260411-0000,masaud_residence,Masaud Residence,Masaud,active,48464.974003,30000.0,18464.974003,61.9,2026-04-29T14:45:27.830435



CLUSTERS


,cluster,cluster_amount_ghs,project_id,quote_id,quoted_total_ghs,client_paid_ghs,remaining_to_collect_ghs,collection_rate_percent
0,cluster_1_pv_mounting,33112.521750,masaud_residence,Q-20260411-0000,48464.974003,30000.0,18464.974003,61.9
1,cluster_2_bos_cable,2467.742023,masaud_residence,Q-20260411-0000,48464.974003,30000.0,18464.974003,61.9
2,cluster_3_cif_general_goods_tema,3103.330230,masaud_residence,Q-20260411-0000,48464.974003,30000.0,18464.974003,61.9
3,cluster_4_cif_battery_goods_tema,5659.500000,masaud_residence,Q-20260411-0000,48464.974003,30000.0,18464.974003,61.9
4,installation_commissioning,3751.440000,masaud_residence,Q-20260411-0000,48464.974003,30000.0,18464.974003,61.9
5,local_transport,370.440000,masaud_residence,Q-20260411-0000,48464.974003,30000.0,18464.974003,61.9



LINE ITEMS


,line_id,description,unit_amount_ghs,quantity,amount_ghs,remarks,category,cluster,project_id,quote_id,client_paid_ghs,collection_rate_percent
0,L1,700W Monofacial Solar Panel,1348.470200,15.000,20227.053000,,pv_module,cluster_1_pv_mounting,masaud_residence,Q-20260411-0000,30000.0,61.9
1,L2,PV mounting system,261.899771,49.200,12885.468750,,mounting_system,cluster_1_pv_mounting,masaud_residence,Q-20260411-0000,30000.0,61.9
2,L3,2 in 2 out DC Combiner (dual MPPT),447.747300,1.000,447.747300,,bos_cable,cluster_2_bos_cable,masaud_residence,Q-20260411-0000,30000.0,61.9
3,L4,2 in 2 out AC Box (Three Phase),482.189400,1.000,482.189400,,bos_cable,cluster_2_bos_cable,masaud_residence,Q-20260411-0000,30000.0,61.9
4,L5,1x6mm² PV Cable (m),847.275660,1.815,1537.805323,,bos_cable,cluster_2_bos_cable,masaud_residence,Q-20260411-0000,30000.0,61.9
5,L6,CIF Shipping of general goods to Tema Port,3103.330230,1.000,3103.330230,,shipping_general_goods,cluster_3_cif_general_goods_tema,masaud_residence,Q-20260411-0000,30000.0,61.9
6,L7,CIF Shipping of battery goods to Tema Port,5659.500000,1.000,5659.500000,,shipping_battery_goods,cluster_4_cif_battery_goods_tema,masaud_residence,Q-20260411-0000,30000.0,61.9
7,L8,Local transport to site,370.440000,1.000,370.440000,,local_transport,local_transport,masaud_residence,Q-20260411-0000,30000.0,61.9
8,L9,Installation & commissioning,3751.440000,1.000,3751.440000,,installation_commissioning,installation_commissioning,masaud_residence,Q-20260411-0000,30000.0,61.9



PAYMENTS


,email_date,project_id,direction,amount_ghs,description,transaction_datetime,balance_after_tx,email_subject,gmail_message_id,source
0,24/04/2026 14:31:47,masaud_residence,cash_in,15000,AL Pay MOHAMMED NAZIF HABIB LTU0000,2026-04-23 13:18,25869.43,Tenet Client Payments,19dbfe7634dfd8bd,gmail_sms
1,24/04/2026 14:31:23,masaud_residence,cash_in,9900,AL Pay MOHAMMED NAZIF HABIB LTU0000,2026-04-23 13:11,10869.43,Tenet Client Payments,19dbfe7067e09eef,gmail_sms
2,24/04/2026 14:30:51,masaud_residence,cash_in,100,AL Pay MOHAMMED NAZIF HABIB LTU0000,2026-04-23 10:20,969.43,Tenet Client Payment,19dbfe689d74ddb8,gmail_sms
3,29/04/2026 13:52:59,masaud_residence,cash_in,5000,AL Pay MOHAMMED NAZIF HABIB LTU0000,2026-04-29 12:10,5640.94,Tenet Client Payments,19dd983af96a34d6,gmail_sms


In [6]:
# ============================================
# MASAUD V2 - BLOCK 05
# CREATE INITIAL ORDERS / PM&P TASK REGISTER
# Writes to same dashboard GSheet: orders
# ============================================


!pip -q install gspread gspread-dataframe


from google.colab import auth
auth.authenticate_user()


import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe
import pandas as pd
from datetime import datetime, timedelta


# ------------------------------------------------
# 1. INPUTS
# ------------------------------------------------
PROJECT_ID = "masaud_residence"
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1cItvZGeYZi0seQEMKSnqhlXgqNlmuMuTJAlKUSGrwAA/edit?gid=1773493460#gid=1773493460"
ORDERS_TAB = "orders"


payment_date = datetime.today().date()
po_due_date = payment_date + timedelta(days=2)


# ------------------------------------------------
# 2. BUILD ORDER TASK REGISTER
# ------------------------------------------------
orders_df = pd.DataFrame([
    {
        "project_id": PROJECT_ID,
        "order_id": "cluster_1_pv_mounting",
        "task_name": "Send PO for Cluster 1 - PV + Mounting",
        "cluster_id": "cluster_1_pv_mounting",
        "supplier": "Alex Sun",
        "amount_due_ghs": None,
        "amount_due_usd": None,
        "due_date": str(po_due_date),
        "status": "pending",
        "status_completion_date": "",
        "dependency": "Client payment received",
        "priority": "high",
        "owner": "Tenet",
        "notes": "PO to be issued based on approved merchant quote"
    },
    {
        "project_id": PROJECT_ID,
        "order_id": "cluster_2_bos_cable",
        "task_name": "Send PO for Cluster 2 - BOS + Cable",
        "cluster_id": "cluster_2_bos_cable",
        "supplier": "CHYT",
        "amount_due_ghs": None,
        "amount_due_usd": None,
        "due_date": str(po_due_date),
        "status": "pending",
        "status_completion_date": "",
        "dependency": "Client payment received",
        "priority": "high",
        "owner": "Tenet",
        "notes": "Awaiting or confirming merchant quotation"
    },
    {
        "project_id": PROJECT_ID,
        "order_id": "cluster_3_general_goods_shipping",
        "task_name": "Send PO for Cluster 3 - General Goods Shipping",
        "cluster_id": "cluster_3_cif_general_goods_tema",
        "supplier": "",
        "amount_due_ghs": None,
        "amount_due_usd": None,
        "due_date": str(po_due_date),
        "status": "pending",
        "status_completion_date": "",
        "dependency": "Cluster 1 and Cluster 2 merchant order confirmation",
        "priority": "medium",
        "owner": "Tenet",
        "notes": "Forwarder PO depends on confirmed goods readiness / shipment route"
    },
    {
        "project_id": PROJECT_ID,
        "order_id": "cluster_4_battery_goods_shipping",
        "task_name": "Send PO for Cluster 4 - Battery Goods Shipping",
        "cluster_id": "cluster_4_cif_battery_goods_tema",
        "supplier": "",
        "amount_due_ghs": None,
        "amount_due_usd": None,
        "due_date": str(po_due_date),
        "status": "pending",
        "status_completion_date": "",
        "dependency": "Battery/inverter merchant order confirmation",
        "priority": "medium",
        "owner": "Tenet",
        "notes": "May not apply immediately if current Masaud scope has no battery/inverter procurement"
    }
])


# ------------------------------------------------
# 3. WRITE TO GSHEET
# ------------------------------------------------
creds, _ = default()
gc = gspread.authorize(creds)


sh = gc.open_by_url(GSHEET_URL)


try:
    ws = sh.worksheet(ORDERS_TAB)
    ws.clear()
except gspread.WorksheetNotFound:
    ws = sh.add_worksheet(title=ORDERS_TAB, rows=100, cols=30)


set_with_dataframe(ws, orders_df, include_index=False)


# ------------------------------------------------
# 4. DISPLAY
# ------------------------------------------------
print("Orders / PM&P task register created.")
print(f"Written to tab: {ORDERS_TAB}")


display(orders_df)



Orders / PM&P task register created.
Written to tab: orders


,project_id,order_id,task_name,cluster_id,supplier,amount_due_ghs,amount_due_usd,due_date,status,status_completion_date,dependency,priority,owner,notes
0,masaud_residence,cluster_1_pv_mounting,Send PO for Cluster 1 - PV + Mounting,cluster_1_pv_mounting,Alex Sun,None,None,2026-05-01,pending,,Client payment received,high,Tenet,PO to be issued based on approved merchant quote
1,masaud_residence,cluster_2_bos_cable,Send PO for Cluster 2 - BOS + Cable,cluster_2_bos_cable,CHYT,None,None,2026-05-01,pending,,Client payment received,high,Tenet,Awaiting or confirming merchant quotation
2,masaud_residence,cluster_3_general_goods_shipping,Send PO for Cluster 3 - General Goods Shipping,cluster_3_cif_general_goods_tema,,None,None,2026-05-01,pending,,Cluster 1 and Cluster 2 merchant order confirm...,medium,Tenet,Forwarder PO depends on confirmed goods readin...
3,masaud_residence,cluster_4_battery_goods_shipping,Send PO for Cluster 4 - Battery Goods Shipping,cluster_4_cif_battery_goods_tema,,None,None,2026-05-01,pending,,Battery/inverter merchant order confirmation,medium,Tenet,May not apply immediately if current Masaud sc...
